# Epsilon Fund — XS Momentum CPCV Validation

Combinatorial Purged Cross-Validation for cross-sectional momentum, via
`infrastructure/walkforward/cpcv_engine.py`.

Run **after** the corresponding walk-forward XS notebook (e.g. `xs_3_i_r`,
`xs_3_2_i_r`) — same panel, same strategy, same universe filter — to get an
unbiased distribution of strategy performance across all `C(N, k)` train/test
splits.

---
### What CPCV gives you (XS context)

| Output | Meaning |
|--------|---------|
| **Path distribution** | 105 complete equity paths (N=8, k=2) — each covers the full dataset once |
| **Mean / Std Sharpe** | Is the dollar-neutral edge reliably positive across history, or just one lucky window? |
| **Parameter stability** | Do `J`, `K`, `pct_long`, `pct_short`, regime EMA/ADX converge across all 28 splits? |
| **Tercile comparison** | Do parameter choices in top-Sharpe splits differ from bottom-Sharpe ones? |
| **Split heatmap** | Which time periods are structurally hard for the L/S signal? |

---
### Workflow

1. Build / refresh the perp universe cache: `python topics/momentum/xs_momentum/universe/universe_cache.py`.
2. Open this notebook, paste your strategy parameter ranges into `PARAM_DEFS`,
   set `FIXED_PARAMS` to lock anything you've already converged on, and run all cells.
3. Set `WF_SHARPE` to the combined OOS Sharpe from the corresponding WF notebook
   (used as the comparison annotation on the path histogram).
4. Save the results pickle for portfolio-level analysis.

---
### Differences from the single-asset CPCV template

- Data is loaded from the **universe cache** (perp Close + Volume + Meta), not a
  single Binance pair fetch.
- A separate BTC OHLC pull provides High/Low for the ADX regime classifier.
- `make_xs_strategy(...)` is a **closure factory** — the strategy_fn it returns
  ignores `df_slice`'s columns and only uses its `DatetimeIndex` to window the
  full panel.  The CPCV engine still calls it with `(df_slice, params)`.
- `position = 1` everywhere → the engine logs zero trades.  The default
  `reject_fn` (which checks `num_trades < 7`) would discard every trial.  The
  custom `reject_fn` below filters on Sharpe / drawdown / total return instead.
- Costs are baked into `strategy_returns` by the strategy_fn → pass `cost=0` to
  the CPCV engine to avoid double-counting.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np


# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'backtester'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum', 'universe'))


from binance_client import get_binance_client, get_data
from cpcv_engine import (
    run_cpcv,
    cpcv_parameter_analysis,
    cpcv_print_param_suggestions,
    cpcv_summary,
    cpcv_confidence_intervals,
    cpcv_ci_summary,
)
from cpcv_visualizer import (
    plot_cpcv_results,
    plot_parameter_distributions,
    plot_parameter_correlation_matrix,
    plot_split_performance_heatmap,
    plot_tercile_comparison,
)
from xs_strategy     import make_xs_strategy, compute_btc_regime_tilt_panel
from ls_diagnostics  import (compute_attribution, plot_attribution,
                             bear_hedge_diagnostic, regime_quadrant_diagnostic)
from universe_filter import load_cache

---
## Data + Universe Selection (FIXED constants)

The four `UF_*` parameters below are **structural design choices**, not
hyperparameters.  They control which coins are eligible to enter the long /
short legs at each rebalance.  Optuna does NOT search over these — keeping the
universe stable across CPCV splits is what makes the parameter-distribution and
tercile diagnostics interpretable.

| Constant | What it controls |
|---|---|
| `UF_TOP_N` | Maximum coins in the eligible universe at each rebalance (ranked by avg volume) |
| `UF_MIN_VOLUME` | Minimum 30-day avg daily USDT perp volume in dollars |
| `UF_MIN_AGE_DAYS` | Minimum days since the perp listing date |
| `UF_VOLUME_WINDOW` | Rolling window for the volume average |

Strategy parameters (`J`, `K`, `pct_long`, `pct_short`, regime EMA/ADX, ...)
go in `PARAM_DEFS` further down and are Optuna-optimisable.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# UNIVERSE FILTER CONFIG  (FIXED — not Optuna-optimisable)
# ══════════════════════════════════════════════════════════════════════════════
UF_TOP_N          = 30          # max coins in the eligible universe at each rebalance
UF_MIN_VOLUME     = 80_000_000  # min 30-day avg daily USDT perp volume ($)
UF_MIN_AGE_DAYS   = 90          # min days since perp listing date
UF_VOLUME_WINDOW  = 30          # rolling window for avg volume computation

# ══════════════════════════════════════════════════════════════════════════════
# DATA WINDOW
# ══════════════════════════════════════════════════════════════════════════════
LOOKBACK = 2200    # days; covers ~6 years of daily bars
COST     = 0.001   # per-leg trading cost fraction (baked into strategy_returns)

# ══════════════════════════════════════════════════════════════════════════════
# Inverse-vol weighting (intra-leg risk parity) — STRUCTURAL constant
# ══════════════════════════════════════════════════════════════════════════════
IV_VOL_WINDOW = 30

# ══════════════════════════════════════════════════════════════════════════════
# Asymmetric tilt magnitude caps — STRUCTURAL constants
# ══════════════════════════════════════════════════════════════════════════════
# Tilt magnitudes are CAPPED here.  Optuna optimises the tilt SCHEDULE
# (regime_ema_period, regime_adx_period, regime_adx_scale) below.
MAX_BULL_TILT = 0.20
MAX_BEAR_TILT = 0.20

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Load multi-asset perp panel from cache + BTC OHLC for regime ADX
# ══════════════════════════════════════════════════════════════════════════════
print('Loading perp cache...')
close_full, volume, meta = load_cache()
print(f'  close  : {close_full.shape}')
print(f'  volume : {volume.shape}')
print(f'  meta   : {meta.shape}')

# Clip price panel to LOOKBACK days; volume + meta intentionally keep full
# history so get_universe's rolling 30-day avg has enough bars even when
# as_of_date sits at the start of LOOKBACK.
cutoff     = close_full.index[-1] - pd.Timedelta(days=LOOKBACK)
close_full = close_full.loc[close_full.index >= cutoff]

# Restrict panel to coins with a recorded listing date
tradeable = sorted([c for c in close_full.columns
                    if c in meta.index and pd.notna(meta.loc[c, 'listing_date'])])
panel = close_full[tradeable]

# Equal-weight basket benchmark + driver_df (BTC, supplies the date index)
ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)
driver_df = panel[['BTC']].rename(columns={'BTC': 'Close'}).copy()

print(f'\nTradeable universe: {len(tradeable)} coins')
print(f'Date range: {panel.index[0].date()} → {panel.index[-1].date()}  '
      f'({len(panel)} bars, clipped to LOOKBACK={LOOKBACK})')

# BTC OHLC for the regime classifier (perp cache only stores Close)
client = get_binance_client()
print('\nFetching BTC OHLC for regime classifier...')
btc_ohlc = get_data(client, 'BTCUSDT', '1d', LOOKBACK)
print(f'  BTC OHLC: {btc_ohlc.shape}  '
      f'({btc_ohlc.index[0].date()} → {btc_ohlc.index[-1].date()})')

---
## Strategy Parameters

Define what Optuna searches (`PARAM_DEFS`) and what's locked (`FIXED_PARAMS`).
**Universe selection is NOT here** — those four constants are fixed above.

The `make_xs_strategy(..., btc_ohlc=...)` factory below makes the regime
EMA / ADX schedule parameters tunable: when `regime_tilt_panel=None` is left
unset, the strategy_fn reads `regime_ema_period`, `regime_adx_period`,
`regime_adx_scale` from `params` each Optuna trial and computes the tilt panel
on the fly (cached by parameter triple, so repeat trials are cheap).

| Param | Type | Suggested range | Notes |
|---|---|---|---|
| `J` | int | 7–45 | Formation / lookback window for the residual-Sharpe signal |
| `K` | int | 7–14 | Holding / rebalance period (shorter = more turnover = more cost) |
| `pct_long` | float | 0.10–0.30 | Fraction of eligible universe in the long leg |
| `pct_short` | float | 0.10–0.30 | Fraction of eligible universe in the short leg |
| `regime_ema_period` | int | 10–40 | BTC EMA span for bull/bear discriminator |
| `regime_adx_period` | int | 7–28 | ADX lookback (Wilder smoothing) |
| `regime_adx_scale` | int | 20–80 | ADX value at which tilt saturates at MAX_*_TILT |

> **Strategy-specific params** (Lee-Swaminathan `pool_multiplier`, IV window
> override, etc.) — fill in below as you decide on the strategy.

In [ ]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
#
# REGIME params are listed here so Optuna can tune the EMA / ADX schedule.
# Move any of them into FIXED_PARAMS once they converge across splits (CV<0.15).
PARAM_DEFS = {
    'J':                 ('int',   7,    45),
    'K':                 ('int',   7,    14),
    'pct_long':          ('float', 0.10, 0.30),
    'pct_short':         ('float', 0.10, 0.30),

    # Regime tilt schedule — all Optuna-optimisable
    'regime_ema_period': ('int',   10,   40),
    'regime_adx_period': ('int',   7,    28),
    'regime_adx_scale':  ('int',   20,   80),
}

# ── anchored params ───────────────────────────────────────────────────────────
# Paste any param here that the corresponding WF notebook converged on (CV<0.15).
# Locking a param removes it from Optuna search and speeds up CPCV by O(N_TRIALS).
FIXED_PARAMS = {
    # 'pct_long': 0.20,
    # 'pct_short': 0.20,
}

---
## Strategy

The XS adapter is a **closure factory**.  `make_xs_strategy(...)` captures the
multi-asset panel + BTC OHLC and returns a `strategy_fn(df_slice, params)` that
the CPCV engine can call exactly like a single-asset strategy.

**Configure the factory** by uncommenting / setting the optional layers below
to whatever your strategy needs:

| Argument | Effect |
|----------|--------|
| `pool_multiplier=None` | Single-stage: rank by composite, pick top/bottom |
| `pool_multiplier=2.0` | Lee-Swaminathan two-stage (composite pool then volume refinement) |
| `volume=volume, meta=meta` | Apply the dynamic universe filter at each rebalance |
| `regime_tilt_panel=...` | Pre-computed (FIXED) regime tilt — skips Optuna regime tuning |
| `btc_ohlc=btc_ohlc` | Per-trial regime tilt computed from `params` — Optuna tunes EMA/ADX |

> **Leave the strategy choice for later** — the line below builds the most
> common variant (dynamic universe + per-trial regime).  Adjust as you go.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Strategy factory (infrastructure/walkforward/xs_strategy.py)
# ══════════════════════════════════════════════════════════════════════════════
my_strategy = make_xs_strategy(
    panel             = panel,
    volume            = volume,
    meta              = meta,
    uf_top_n          = UF_TOP_N,
    uf_min_volume     = UF_MIN_VOLUME,
    uf_min_age        = UF_MIN_AGE_DAYS,
    uf_volume_window  = UF_VOLUME_WINDOW,
    # pool_multiplier = 2.0,         # ← uncomment for Lee-Swaminathan two-stage
    iv_vol_window     = IV_VOL_WINDOW,

    # Per-trial regime path (Optuna tunes EMA / ADX / scale).
    # Replace with `regime_tilt_panel = compute_btc_regime_tilt_panel(...)` to
    # FIX the regime and skip those three Optuna params.
    btc_ohlc          = btc_ohlc,
    max_bull_tilt     = MAX_BULL_TILT,
    max_bear_tilt     = MAX_BEAR_TILT,
)

---
## CPCV Config + Scoring

Partitions the data into `N` equal groups and optimises on every `C(N, k)`
training complement.  Each split produces an OOS evaluation on the `k` held-out
groups.  All groups stitch into 105 complete equity paths (for N=8, k=2).

| Parameter | Description | Default |
|-----------|-------------|---------|
| `N` | Groups — 8 gives C(8,2)=28 splits and 105 complete paths | `8` |
| `K_HOLD` | Groups held out per split | `2` |
| `N_TRIALS` | Optuna trials per split — match the WF notebook for consistency | `300` |
| `BURNIN` | Bars prepended to each test group for indicator warmup | `95` |
| `PURGE_BARS` | Training bars purged at each train/test boundary | `1` |
| `COST` | Per-leg trading cost fraction (already baked into `strategy_returns`) | `0.001` |

**Burnin = 95** is the same value the xs_3 family WF notebooks use — the
residual-Sharpe signal does TWO compounded `rolling(J)` operations (rolling β,
then rolling Sharpe of residuals), so warmup is `2·max(J) + 1`.  With max
`J=45` you need ≥ 91; 95 leaves a small safety margin.

> **Runtime:** `C(N, k) × N_TRIALS` Optuna evaluations.
> For N=8, k=2, N_TRIALS=300: ~8,400 backtests — expect 30–90 min on daily data.

In [ ]:
N          = 8     # groups to partition data into  →  C(8,2) = 28 splits
K_HOLD     = 2     # groups held out per split
N_TRIALS   = 300   # Optuna trials per split (match WF notebook)
BURNIN     = 95    # 2·max(J) + 1 for the residual-Sharpe signal at max J=45
PURGE_BARS = 1

# ── walk-forward Sharpe for comparison annotation (optional) ──────────────────
WF_SHARPE  = None   # ← set to combined OOS Sharpe from the WF notebook


# ── SCORING FUNCTION ─────────────────────────────────────────────────────────
# Match the WF notebook's score_fn so CPCV ranks splits the same way the WF
# folds were ranked — keeps parameter-distribution / tercile output comparable.
def SCORE_FN(m):
    SHARPE_MAX = 3.0
    CALMAR_MAX = 6.0
    RETURN_MAX = 10.0
    calmar = (m['total_return'] / abs(m['max_drawdown'])
              if m['max_drawdown'] != 0 else 0.0)
    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)
    return 0.60 * s + 0.25 * c + 0.15 * r


# ── REJECTION CRITERIA ───────────────────────────────────────────────────────
# Default reject_fn checks num_trades < 7 → would reject every trial because
# this strategy uses position=1 everywhere (precomputed-returns path).  Filter
# on Sharpe / drawdown / total return instead.
def REJECT_FN(m):
    if m is None:                   return True
    if m['sharpe_ratio'] < 0.0:     return True
    if m['max_drawdown'] < -0.80:   return True
    if m['total_return'] < 0.0:     return True
    return False

In [ ]:
results = run_cpcv(
    df           = driver_df,         # BTC supplies the calendar; strategy_fn ignores its columns
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    N            = N,
    k            = K_HOLD,
    purge_bars   = PURGE_BARS,
    n_trials     = N_TRIALS,
    burnin       = BURNIN,
    cost         = 0.0,               # costs already baked into strategy_returns
    score_fn     = SCORE_FN,
    reject_fn    = REJECT_FN,
    verbose      = True,
)

### CPCV Summary — display flags

`cpcv_summary` is split into two cells so the output stays readable. Each flag
controls one section of the printed report.

| Flag | Default | What it prints |
|---|---|---|
| `show_group_legend` | `True` | **Group date ranges** — maps each group number (g1–g8) to its calendar window. |
| `show_distribution` | `True` | **Path distribution table** — Mean / Std / Min / Q25 / Median / Q75 / Max / IQR across all 105 paths. |
| `show_paths` | `False` | **Full per-path table** — verbose; turn on when auditing individual paths. |
| `show_highlights` | `True` | **Top 5 / Bottom 5 paths** with their group→split assignments. |
| `show_split_legend` | `False` | **Split legend** — decodes `gN→sM` notation. |
| `show_ci` | `True` | **Confidence intervals** — overlap-adjusted CI for Sharpe, Calmar, Max DD. |

In [ ]:
# Group legend + path distribution percentile table + CI
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = True,
    show_paths        = False,
    show_highlights   = False,
    show_split_legend = False,
    show_ci           = True,
)

In [ ]:
# Top/bottom highlights + split legend
cpcv_summary(
    results,
    show_group_legend = False,
    show_distribution = False,
    show_paths        = False,
    show_highlights   = True,
    show_split_legend = True,
    show_ci           = False,
)

---
## Path-Level Charts

- **Equity curves** — all 105 paths (semi-transparent), the mean path (bold),
  and the min–max envelope.  Vertical dotted lines mark group boundaries.
- **Sharpe histogram** — distribution of the 105 path-level Sharpe ratios with
  the overlap-adjusted CI overlay.

The CI overlay uses an effective sample size `N_eff = N² / sum(overlap_matrix)`
— because CPCV paths share splits, the 105 paths are NOT independent, so
`N_eff` is substantially smaller than 105.  The conservative lower bound is
the most cautious point estimate of mean Sharpe — use it when comparing
against a hurdle rate.

In [ ]:
ci = cpcv_confidence_intervals(results)
plot_cpcv_results(results, wf_sharpe=WF_SHARPE, ci_results=ci);

---
## Parameter Analysis

Compute the analysis object once — all four charts below read from it.

In [ ]:
analysis = cpcv_parameter_analysis(results)

# Prints consensus_ranges table + copy-pasteable PARAM_DEFS / FIXED_PARAMS blocks.
# CV < 0.10  → "fix at {median}"   : converges consistently, safe to anchor.
# CV 0.10–0.25 → "narrow to IQR"  : reduce search range to Q25–Q75 next CPCV run.
# CV > 0.25  → "keep current range": period-sensitive, keep free.
cpcv_print_param_suggestions(results, analysis)

### Parameter Distributions

One subplot per free parameter.  Each dot is one split's best value, jittered
horizontally for readability and **coloured by that split's OOS Sharpe**
(red = low, green = high).

For XS specifically: the regime params (`regime_ema_period`,
`regime_adx_period`, `regime_adx_scale`) are good candidates to inspect first.
If they cluster tightly around stable values, fix them and re-run with
narrower / fewer free dimensions.

In [ ]:
plot_parameter_distributions(results, analysis=analysis);

### Parameter Correlation Matrix

Pearson correlation between every pair of free parameters across the 28
splits.  |r| > 0.6 means the optimizer is sliding along a ridge — consider
fixing the ratio or removing one parameter.

In [ ]:
plot_parameter_correlation_matrix(analysis);

### Split Performance Heatmap

N × N grid where cell (i, j) shows the OOS Sharpe of the split that held out
groups i and j.  Persistent red cells around a particular group index means
that calendar period is structurally hard for the L/S signal — cross-reference
the group date legend to identify the regime.

In [ ]:
plot_split_performance_heatmap(results);

### Tercile Comparison

Each subplot: parameter values chosen by the top vs bottom Sharpe tercile of
splits.  Separation > 1 means parameter choice is predictive of OOS
performance — anchor toward the top-tercile median or narrow the search range.

In [ ]:
plot_tercile_comparison(results, analysis);

---
## XS-Specific Diagnostics — Consensus-Param Backtest

Re-run the strategy on the FULL date range using the consensus parameters
(median across CPCV splits) and compute the long/short attribution + bear-hedge
+ regime-quadrant diagnostics.  These are the same OOS diagnostics the WF
notebooks print, but applied to the cross-validated consensus rather than to a
single WF schedule.

Skip this section if you only care about CPCV path statistics — it adds one
strategy_fn call beyond the CPCV run itself.

In [ ]:
consensus = {**FIXED_PARAMS, **results['consensus_params']}
print('Consensus params:')
for k, v in consensus.items():
    print(f'  {k:<22}  {v}')

# Build a full-range slice with the same OHLC shape strategy_fn expects
full_slice = driver_df.copy()
oos_full, _ = my_strategy(full_slice, consensus)

# Trim warmup so the diagnostic only sees bars where the strategy was active
oos = oos_full.iloc[BURNIN:].copy()
print(f'\nConsensus-path OOS span: {oos.index[0].date()} → {oos.index[-1].date()}  '
      f'({len(oos)} bars)')

In [ ]:
attr = compute_attribution(oos, ann=365)
print('═' * 60)
print('LONG/SHORT ATTRIBUTION (consensus path)')
print('═' * 60)
for k, v in attr.items():
    if isinstance(v, float):
        if 'sharpe' in k:
            print(f'  {k:<22} {v:>8.3f}')
        elif 'return' in k or 'total' in k or 'rate' in k or 'turnover' in k:
            print(f'  {k:<22} {v:>8.2%}' if abs(v) < 100 else f'  {k:<22} {v:>8.3f}')
        else:
            print(f'  {k:<22} {v:>8.2f}')
    else:
        print(f'  {k:<22} {v:>8}')

plot_attribution(oos, benchmark_data=ew_df, show=True, save_html=None,
                 title='XS Momentum — L/S Diagnostics (CPCV consensus path)')

In [ ]:
bh = bear_hedge_diagnostic(oos, panel['BTC'], ann=365)

In [ ]:
rq = regime_quadrant_diagnostic(oos, panel['BTC'],
                                ma_window=200, er_window=14, ann=365)

---
## Save Results

Pickle the full results dict for portfolio-level analysis and future reference.

In [ ]:
os.makedirs('oos', exist_ok=True)
results_file = os.path.join('oos', 'xs_cpcv.pkl')
pd.to_pickle(results, results_file)
print(f'Saved CPCV results to {results_file}')

# Optional: save the consensus-path OOS dataframe for portfolio integration
oos_file = os.path.join('oos', 'xs_cpcv_consensus_oos.pkl')
oos.to_pickle(oos_file)
print(f'Saved consensus-path OOS dataframe to {oos_file}')